# 01 · 感知机 Perceptron —— 深度学习的起点

**家族位置**：`01_Fundamentals_MLP` 第 1 个项目（后续：`02_MLP_MNIST` → `03_Training_Tricks_Ablation`）

**任务背景**：感知机是 Rosenblatt 在 1957 年提出的最早的可学习分类模型，是所有神经网络的"细胞"。本项目用 NumPy **手写**感知机，理解它**能解决什么、不能解决什么**——后者正是深度学习存在的理由。

**学习目标**
1. 理解感知机模型与误分类驱动的学习规则
2. NumPy 手写实现，在 3 组数据上验证
3. 用 sklearn 对照，确认实现正确
4. 通过 XOR / 双半月失效案例理解"线性可分"限制，引出 MLP

## 1. 感知机原理

### 模型

二分类线性模型，标签约定 $y \in \{-1, +1\}$：

$$f(x) = \mathrm{sign}(w^\top x + b)$$

### 学习规则（误分类驱动）

逐个样本检查，凡是**被误分**的点 $(x_i, y_i)$，即满足 $y_i(w^\top x_i + b) \le 0$，立即更新：

$$w \leftarrow w + \eta\, y_i x_i, \qquad b \leftarrow b + \eta\, y_i$$

直观理解：真实标签是 $+1$ 却被判成负类 → 把 $w$ 向 $x_i$ 方向挪一步；反之反向挪。

### 损失函数视角

感知机的经验损失 = 所有误分类点到超平面的总距离：

$$L(w, b) = -\sum_{x_i \in M} y_i (w^\top x_i + b)$$

对它做**随机梯度下降**，恰好得到上面的学习规则——"逐样本纠错"和"优化损失"是同一件事。

### 收敛性（Novikoff 定理）

若数据**线性可分**，误分类次数有上界 $(R/\gamma)^2$（$R$ 为点到原点最大距离，$\gamma$ 为间隔），算法必在有限步内收敛；**若线性不可分，$w$ 会震荡、永不收敛**——这正是后面 XOR 实验将看到的。

## 2. 准备工作

公共代码已抽到家族模块 `common/`（`data.py` 数据、`models.py` 模型、`utils.py` 工具），notebook 只做实验与可视化——这是本仓库的统一规范。

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录，请确认项目结构完整"
sys.path.insert(0, str(ROOT))

from common.data import load_iris_binary, make_linearly_separable, make_xor, train_test_split
from common.models import Perceptron
from common.utils import plot_data, plot_decision_boundary, set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
print("环境就绪：torch", torch.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")

### 2.1 数据一：线性可分二维数据（成功案例）

`make_linearly_separable` 生成两团**严格线性可分**的高斯点（构造上保证硬间隔）。注意家族数据约定标签为 `{0,1}`，而感知机的经典约定是 `{-1,+1}`，使用前需转换——这也是初学时最容易踩的坑之一。

In [ ]:
X, y, _ = make_linearly_separable(n=200, seed=0)
Xn, yn = X.numpy(), np.where(y.numpy() == 0, -1, 1)  # {0,1} → {-1,+1}

plot_data(Xn, yn, title="线性可分数据：两团高斯（硬间隔）")
plt.show()
print("X shape:", Xn.shape, "| 标签取值:", sorted(set(yn)))

### 2.2 手写感知机：先看核心逻辑

完整实现位于 `common/models.py`（家族复用），核心只有三行：

```python
if y[i] * (X[i] @ w + b) <= 0:   # 误分类判定
    w += lr * y[i] * X[i]        # 向正确方向拉一步
    b += lr * y[i]
```

完整版本在此基础上增加了：**随机洗牌**（SGD）、**epoch 循环**、**收敛检测**（一整轮零误分即停）、**训练统计**（更新次数 / 轮数）。接口与 sklearn 风格一致：`fit / predict / score`。

In [ ]:
# 直观演示：单个误分点如何"拉"动 w
w, b, lr = np.zeros(2), 0.0, 1.0
i = int(np.where(yn == 1)[0][0])  # 取一个正类点
print(f"更新前: w = {w}, b = {b}")
print(f"第 {i} 个样本 (+1 类) 的 margin = y·(w·x+b) = {yn[i] * (Xn[i] @ w + b):.1f} ≤ 0 → 误分")
w, b = w + lr * yn[i] * Xn[i], b + lr * yn[i]
print(f"更新后: w = {w}, b = {b:.2f}   （w 被拉向了该正类点）")

In [ ]:
clf = Perceptron(lr=1.0, max_epochs=100, seed=0).fit(Xn, yn)
print(f"收敛: {clf.converged_} | 轮数: {clf.n_epochs_} | 参数更新次数: {clf.n_updates_}")
print(f"w = {clf.w_}, b = {clf.b_:.2f}")
print(f"训练集准确率: {clf.score(Xn, yn):.2%}")

In [ ]:
plot_decision_boundary(clf, Xn, yn, title=f"感知机决策边界（{clf.n_epochs_} 轮收敛）")
plt.savefig(FIGS / "fig1_linear.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.3 数据二：Iris 鸢尾花二分类（真实数据）

取经典 Iris 数据中的两类（setosa / versicolor），只用**花瓣长、花瓣宽**两个特征——在这个特征空间里两类线性可分。25% 作测试集。

In [ ]:
X, y, _ = load_iris_binary()  # y 已是 {-1,+1}
print("特征: petal length (cm), petal width (cm) | +1 = setosa, -1 = versicolor")

Xtr, ytr, Xte, yte = train_test_split(X, y, test_ratio=0.25, seed=0)

clf_iris = Perceptron(lr=1.0, max_epochs=100, seed=0).fit(Xtr.numpy(), ytr.numpy())
print(f"收敛: {clf_iris.converged_} | 轮数: {clf_iris.n_epochs_} | 更新次数: {clf_iris.n_updates_}")
print(f"测试集准确率: {clf_iris.score(Xte.numpy(), yte.numpy()):.2%}")

plot_decision_boundary(clf_iris, X.numpy(), y.numpy(), title="Iris 二分类决策边界（全量数据着色）")
plt.savefig(FIGS / "fig3_iris.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.4 sklearn 对照验证

手写实现最怕"自说自话"。用 `sklearn.linear_model.Perceptron` 在同一划分上对照，两者一致才能说明实现无误。

In [ ]:
from sklearn.linear_model import Perceptron as SkPerceptron

sk = SkPerceptron(max_iter=1000, tol=None, random_state=0).fit(Xtr.numpy(), ytr.numpy())
print(f"手写实现 测试准确率: {clf_iris.score(Xte.numpy(), yte.numpy()):.2%}")
print(f"sklearn   测试准确率: {sk.score(Xte.numpy(), yte.numpy()):.2%}")
print("结论：手写实现与工业级实现一致 ✔")

### 2.5 失效案例：XOR（线性不可分）

XOR 的四个象限：同号象限为一类、异号象限为另一类。**任何一条直线都无法把它分开**（两类的代表点在对角位置上互相"包围"）。感知机在这里会永久震荡。

In [ ]:
X, y, _ = make_xor(n=300, noise=0.15, seed=0)
Xn, yn = X.numpy(), np.where(y.numpy() == 0, -1, 1)

clf_xor = Perceptron(lr=1.0, max_epochs=200, seed=0).fit(Xn, yn)
print(f"收敛: {clf_xor.converged_} | 训练 200 轮后准确率: {clf_xor.score(Xn, yn):.2%}")
print("→ w 在多个次优解之间震荡，永远无法收敛到 100%")

plot_decision_boundary(clf_xor, Xn, yn, title=f"XOR：线性天花板约 50%，感知机卡在 {clf_xor.score(Xn, yn):.2%}")
plt.savefig(FIGS / "fig2_xor.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. 误差分析

| 数据集 | 线性可分？ | 感知机表现 | 原因 |
|---|---|---|---|
| 双高斯团簇 | ✔ | 2 轮收敛，100% | 有硬间隔，Novikoff 定理保证有限步收敛 |
| Iris（两特征） | ✔ | 2 轮收敛，测试 100% | 特征空间中两类完全分离 |
| XOR | ✘ | 不收敛，卡在 ~56% | 任何直线最多分对约一半，$w$ 在次优解间震荡 |

**共性结论**：感知机的假设空间只有"直线"（超平面）。当正确答案不在假设空间里时，再怎么更新参数也无济于事——这不是**优化问题**，而是**模型表达能力**问题。优化再努力，也学不出假设空间里不存在的东西。

（补充：`make_two_moons` 双半月数据是更真实的线性不可分案例，思路相同，可自行在 `common/data.py` 中扩展练习。）

## 4. 总结与改进方向

**本项目收获**
1. 感知机 = 线性模型 + 误分类驱动的 SGD；线性可分时有限步收敛（且收敛可能快得出乎意料——本项目只用了 1~13 次更新）
2. 失效根源是**假设空间受限**（线性），不是优化失败 → 深度学习的动机
3. 工程细节：标签约定（±1 vs 0/1）、随机洗牌、收敛检测、sklearn 对照验证——这些习惯会贯穿整个深度学习系列

**下一步（引出 MLP）**
把多个感知机叠成两层：第一层学出多条"线性切分"，第二层用非线性激活组合它们——就能拼出 XOR、双半月这类非线性边界。这就是**多层感知机 MLP**：

- `02_MLP_MNIST`：用 PyTorch 实现 MLP，在 MNIST 上做到 98%+
- `03_Training_Tricks_Ablation`：优化器 / 归一化 / 正则化系统消融

**延伸阅读**：Pocket 算法（不可分时保留历史最优 $w$）、投票感知机、把 sign 换成 sigmoid 就得到 Logistic 回归（概率化视角）